In [1]:
from pyspark import SparkContext

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.getOrCreate()

In [4]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [5]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [14]:
from pyspark.sql.functions import min as _min, max as _max

# TWÓJ KOD
categories = (df.groupBy("category")
              .agg(
                  _round(_sum("amount"), 2).alias("suma_PLN"),
                  _min("amount").alias("min_kwota"),
                  _max("amount").alias("max_kwota")
            )
            .orderBy("category").show())

+-----------+----------+---------+---------+
|   category|  suma_PLN|min_kwota|max_kwota|
+-----------+----------+---------+---------+
|elektronika|1520770.69|      9.0|   9999.0|
|    książki| 851382.08|      5.0|  9107.25|
|     odzież| 849877.55|      5.0|  9696.63|
|    żywność| 789514.43|      5.0|  6916.92|
+-----------+----------+---------+---------+



In [8]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [9]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [10]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [17]:
# TWÓJ KOD
(df.groupBy(window("timestamp", "30 minutes"), "store")
   .agg(
       count("tx_id").alias("liczba_tx"),
       _round(_sum("amount"), 2).alias("suma_PLN")
   )
   .select(
       col("window.start").alias("od"),
       col("window.end").alias("do"),
       "liczba_tx",
       "suma_PLN",
       "store"
   )
   .orderBy("od", "store")
   .show())

+-------------------+-------------------+---------+---------+--------+
|                 od|                 do|liczba_tx| suma_PLN|   store|
+-------------------+-------------------+---------+---------+--------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|      252| 93391.22|  Gdańsk|
|2026-04-12 08:00:00|2026-04-12 08:30:00|      289|117786.42|  Kraków|
|2026-04-12 08:00:00|2026-04-12 08:30:00|      275| 88441.58|Warszawa|
|2026-04-12 08:00:00|2026-04-12 08:30:00|      296|111540.59| Wrocław|
|2026-04-12 08:30:00|2026-04-12 09:00:00|      514|209187.85|  Gdańsk|
|2026-04-12 08:30:00|2026-04-12 09:00:00|      532|223541.41|  Kraków|
|2026-04-12 08:30:00|2026-04-12 09:00:00|      490|182435.06|Warszawa|
|2026-04-12 08:30:00|2026-04-12 09:00:00|      502|215587.17| Wrocław|
|2026-04-12 09:00:00|2026-04-12 09:30:00|      619|253364.95|  Gdańsk|
|2026-04-12 09:00:00|2026-04-12 09:30:00|      590|224358.03|  Kraków|
|2026-04-12 09:00:00|2026-04-12 09:30:00|      584|214573.66|Warszawa|
|2026-

In [23]:
from pyspark.sql.functions import desc

# TWÓJ KOD

(df.groupBy(window("timestamp", "1 hour"), "store")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .filter(col("store") == "Kraków")
    .orderBy(desc("suma_PLN"))
    .show())

+--------------------+------+---------+
|              window| store| suma_PLN|
+--------------------+------+---------+
|{2026-04-12 09:00...|Kraków|483309.86|
|{2026-04-12 08:00...|Kraków|341327.83|
|{2026-04-12 10:00...|Kraków|201259.26|
+--------------------+------+---------+



In [11]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [12]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")


Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [27]:
# PD 2
#Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
(df.groupBy(window("timestamp", "1 hour"), "store")
    .agg(
        _round(avg("amount"), 2).alias("srednia_suma")
    )
    .select(
       col("window.start").alias("od"),
       col("window.end").alias("do"),
        "srednia_suma",
        "store"
   )
    .filter(col("store") == "Gdańsk")
    .orderBy("srednia_suma")
    .show())

+-------------------+-------------------+------------+------+
|                 od|                 do|srednia_suma| store|
+-------------------+-------------------+------------+------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|      395.01|Gdańsk|
|2026-04-12 10:00:00|2026-04-12 11:00:00|      412.92|Gdańsk|
|2026-04-12 09:00:00|2026-04-12 10:00:00|      415.91|Gdańsk|
+-------------------+-------------------+------------+------+



In [38]:
#Policz ile transakcji per kategoria było w oknie 09:00–09:30.
(df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "category"
    )
    .filter(
        (col("od") >= "2026-04-12 09:00:00") & 
        (col("do") <= "2026-04-12 09:30:00")
    )
    .orderBy("od")
    .show())

+-------------------+-------------------+---------+-----------+
|                 od|                 do|liczba_tx|   category|
+-------------------+-------------------+---------+-----------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|      611|elektronika|
|2026-04-12 09:00:00|2026-04-12 09:30:00|      605|     odzież|
|2026-04-12 09:00:00|2026-04-12 09:30:00|      622|    książki|
|2026-04-12 09:00:00|2026-04-12 09:30:00|      567|    żywność|
+-------------------+-------------------+---------+-----------+



In [48]:
#Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).
(df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("max_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "max_tx"
    )
    .orderBy(desc("max_tx"))
    .show())


+-------------------+-------------------+------+
|                 od|                 do|max_tx|
+-------------------+-------------------+------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|  1234|
|2026-04-12 09:00:00|2026-04-12 09:15:00|  1171|
|2026-04-12 09:30:00|2026-04-12 09:45:00|  1156|
|2026-04-12 08:45:00|2026-04-12 09:00:00|  1139|
|2026-04-12 09:45:00|2026-04-12 10:00:00|  1100|
|2026-04-12 08:30:00|2026-04-12 08:45:00|   899|
|2026-04-12 10:00:00|2026-04-12 10:15:00|   858|
|2026-04-12 08:15:00|2026-04-12 08:30:00|   644|
|2026-04-12 10:15:00|2026-04-12 10:30:00|   582|
|2026-04-12 08:00:00|2026-04-12 08:15:00|   468|
|2026-04-12 10:30:00|2026-04-12 10:45:00|   443|
|2026-04-12 10:45:00|2026-04-12 11:00:00|   306|
+-------------------+-------------------+------+

